# 🏋️ Day 4 — Workbook: Text Preprocessing, Chunking & Retrieval
### Course: Python for GenAI & Data Analysis | Date: 6 April 2026

---

> **How to use this workbook:**  
> Each section has 🧠 **concept questions**, 🔧 **coding exercises**, and 🚀 **challenge tasks**.  
> Fill in the `# YOUR CODE HERE` blocks. Run each cell to check your work.
> Difficulty: ⭐ Easy | ⭐⭐ Medium | ⭐⭐⭐ Hard

---

## Setup — Run This First

In [ ]:
import re, math, string, numpy as np
from collections import Counter, defaultdict
from typing import List, Tuple, Dict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

# ─── Shared utilities (same as notes notebook) ───────────────────────────────
STOP_WORDS = {
    'a','an','the','and','or','but','in','on','at','to','for','of','with',
    'by','from','up','about','into','through','during','is','are','was',
    'were','be','been','being','have','has','had','do','does','did','will',
    'would','could','should','may','might','shall','can','need','dare',
    'ought','used','it','its','this','that','these','those','i','you','he',
    'she','we','they','what','which','who','whom','not','no','so','if',
    'as','than','then','there','their','they','our','your','his','her',
    'my','me','him','us','them','each','few','more','most','other','some',
    'such','any','only','same','also','just','because','while','how','all'
}

def simple_sent_tokenize(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]

def simple_stem(word):
    for sfx in ['ing','tion','ness','ment','ly','er','ed','es','s']:
        if word.endswith(sfx) and len(word)-len(sfx) > 3:
            return word[:-len(sfx)]
    return word

# ─── Grading helper ──────────────────────────────────────────────────────────
def check(condition, msg_pass, msg_fail):
    if condition:
        print(f"  ✅ PASS — {msg_pass}")
    else:
        print(f"  ❌ FAIL — {msg_fail}")
    return condition

print("✅ Setup complete! Let's get started.")

---
# 📋 Part 1 — Conceptual Questions

Answer these questions in the markdown cells provided. Write your answers **replacing** the `YOUR ANSWER HERE` text.

### Q1 ⭐ — Why do we need text preprocessing?

In 2–3 sentences, explain why preprocessing is necessary before feeding text into a retrieval or NLP system.

**Your Answer:**

> YOUR ANSWER HERE

### Q2 ⭐ — Stemming vs Lemmatization

Fill in the table:

| Word | Stemmed (guess) | Lemmatized (correct) |
|------|-----------------|----------------------|
| running | ? | ? |
| better | ? | ? |
| studies | ? | ? |
| geese | ? | ? |
| was | ? | ? |

### Q3 ⭐⭐ — When NOT to remove stop words

List **3 specific scenarios** where removing stop words would **hurt** your system's performance. Be specific about why.

**Your Answer:**

1. YOUR ANSWER HERE
2. YOUR ANSWER HERE  
3. YOUR ANSWER HERE

### Q4 ⭐⭐ — Chunking Strategy Choice

For each use case below, which chunking strategy would you choose and why?

| Use Case | Best Strategy | Why? |
|----------|--------------|------|
| A 200-page legal contract PDF | ? | ? |
| A chatbot FAQ with 500 short entries | ? | ? |
| A Python code repository (50 files) | ? | ? |
| A news article Q&A system | ? | ? |
| Medical research papers for RAG | ? | ? |

### Q5 ⭐⭐ — TF-IDF Intuition

Consider a corpus of 1000 news articles. The word **"the"** appears in all 1000 articles. The word **"Quantum"** appears in only 3 articles.

- What is the IDF of "the"?
- What is the IDF of "Quantum"?
- Which word will have a higher TF-IDF score in one of those 3 articles?
- **Why** is this the desired behavior for a retrieval system?

*Hint: IDF = log(N / df(t))*

**Your Answer:**

> YOUR ANSWER HERE

### Q6 ⭐⭐⭐ — BM25 Term Saturation

BM25 adds **term saturation** — meaning the 10th occurrence of a word adds much less value than the 1st occurrence.

- **Why** is this desirable? What problem does it solve compared to raw TF?
- If you were writing a spam email trying to rank highly for the query "cheap iphone", how might term saturation hurt your spam strategy?
- What is the BM25 `k1` parameter and what happens if you set it to `k1=0` vs `k1=100`?

**Your Answer:**

> YOUR ANSWER HERE

### Q7 ⭐⭐⭐ — RAG Architecture

Draw (describe in text) the complete flow of a RAG pipeline from user query to final answer. Include:
- What happens at each stage
- What goes IN and what comes OUT of each stage
- Where keyword retrieval fits vs where an LLM fits

**Your Answer:**

```
User Query
    │
    ▼
[Stage 1: ???] 
    │ Output: ???
    ▼
[Stage 2: ???]
    ...
```

*Fill in the stages above*

---
# 🔧 Part 2 — Coding Exercises

Fill in the `# YOUR CODE HERE` sections.

## Exercise 1 ⭐ — Basic Text Cleaning

Write a function `clean_tweet(text)` that:
1. Lowercases the text
2. Removes URLs (starting with `http` or `www`)
3. Removes @mentions (e.g., `@username`)
4. Removes hashtags (e.g., `#AI`)
5. Removes extra whitespace

In [ ]:
def clean_tweet(text: str) -> str:
    """
    Clean a tweet by removing URLs, mentions, hashtags, and normalizing spaces.
    
    Examples:
    >>> clean_tweet("@John_Doe: Check out https://example.com #MachineLearning is amazing!")
    'check out is amazing!'
    """
    # Step 1: Lowercase
    # YOUR CODE HERE
    
    # Step 2: Remove URLs (http..., www...)
    # YOUR CODE HERE
    
    # Step 3: Remove @mentions
    # YOUR CODE HERE
    
    # Step 4: Remove #hashtags
    # YOUR CODE HERE
    
    # Step 5: Normalize whitespace
    # YOUR CODE HERE
    
    return text


# ─── Test cases ───────────────────────────────────────────────────────────────
tests = [
    (
        "@JohnDoe Check out https://openai.com #MachineLearning is amazing! 🚀",
        "check out is amazing! 🚀"
    ),
    (
        "Just deployed my #RAG pipeline! www.myblog.com @mentor_paul",
        "just deployed my pipeline!"
    ),
    (
        "No special content here.",
        "no special content here."
    )
]

print("Testing clean_tweet():")
all_pass = True
for i, (inp, expected) in enumerate(tests):
    result = clean_tweet(inp)
    # Normalize spaces for comparison
    result_norm   = re.sub(r'\s+', ' ', result).strip()
    expected_norm = re.sub(r'\s+', ' ', expected).strip()
    passed = result_norm == expected_norm
    all_pass = all_pass and passed
    status = "✅" if passed else "❌"
    print(f"  {status} Test {i+1}:")
    print(f"     Input:    {inp[:60]}")
    print(f"     Expected: {expected_norm}")
    print(f"     Got:      {result_norm}")

print(f"\n{'🎉 All tests passed!' if all_pass else '⚠️  Some tests failed — review your regex patterns'}")

## Exercise 2 ⭐ — Token Frequency Analysis

Given a text, find the top-N most frequent **meaningful** words (after removing stop words and punctuation).

In [ ]:
def top_n_words(text: str, n: int = 10) -> List[Tuple[str, int]]:
    """
    Return the top-N most frequent meaningful words in text.
    
    Steps:
    1. Lowercase
    2. Extract only alphabetic tokens (re.findall)
    3. Remove stop words
    4. Remove words shorter than 3 characters
    5. Count frequencies
    6. Return top N as (word, count) tuples
    """
    # YOUR CODE HERE
    pass


corpus_text = """
Artificial intelligence and machine learning are transforming industries worldwide.
Machine learning algorithms learn patterns from data to make predictions.
Deep learning, a subset of machine learning, uses neural networks with many layers.
Natural language processing, another branch of AI, helps computers understand human language.
Large language models represent a breakthrough in natural language processing capabilities.
These models are trained on massive text datasets using deep learning techniques.
"""

result = top_n_words(corpus_text, n=10)

print("Top 10 most frequent meaningful words:")
if result:
    for word, count in result:
        bar = '█' * count
        print(f"  {word:<20} {count:>3}  {bar}")
else:
    print("  ❌ Function returned None or empty — implement the function!")

# Auto-check
if result:
    top_words = [w for w, c in result]
    check('learning' in top_words,  "'learning' in top words",  "'learning' should be in top words")
    check(len(result) == 10,        "Returns exactly 10 words",  "Should return exactly N=10 words")
    check(result[0][1] >= result[-1][1], "Sorted by frequency", "Should be sorted descending")

## Exercise 3 ⭐⭐ — Token-Based Chunking

Implement a chunker that splits text by **word count** (not character count), with overlap.

In [ ]:
def token_chunk(
    text: str,
    max_tokens: int = 50,
    overlap_tokens: int = 10
) -> List[Dict]:
    """
    Split text into chunks of max_tokens words, with overlap_tokens overlap.
    
    Parameters:
    -----------
    text           : Input text
    max_tokens     : Maximum words per chunk
    overlap_tokens : Number of words shared between consecutive chunks
    
    Returns:
    --------
    List of dicts: {'id', 'text', 'token_count', 'start_token', 'end_token'}
    
    Hints:
    - First split text into words: words = text.split()
    - Slide a window of size max_tokens with step = max_tokens - overlap_tokens
    - Join each window back into a string
    """
    # YOUR CODE HERE
    pass


test_text = ("The field of artificial intelligence encompasses machine learning, deep learning, "
             "natural language processing, computer vision, robotics, and many other subfields. "
             "Researchers are constantly pushing the boundaries of what machines can do. "
             "Modern AI systems can write code, generate images, answer complex questions, "
             "and even engage in creative tasks that were once thought to require human intelligence. "
             "The rapid advancement of large language models has particularly transformed "
             "how we interact with computers and access information.")

chunks = token_chunk(test_text, max_tokens=20, overlap_tokens=5)

print("Token-Based Chunking (max=20 tokens, overlap=5):")
if chunks:
    for c in chunks:
        print(f"  Chunk {c['id']} [{c['start_token']}:{c['end_token']}] ({c['token_count']} tokens): {c['text'][:70]}...")
    
    # Auto-check
    print()
    check(all(c['token_count'] <= 20 for c in chunks), 
          "All chunks ≤ 20 tokens", 
          "Some chunks exceed max_tokens!")
    check(len(chunks) > 1, 
          f"Multiple chunks created ({len(chunks)})", 
          "Should create multiple chunks")
    # Check overlap: last N tokens of chunk[i] should appear at start of chunk[i+1]
    if len(chunks) >= 2:
        words0 = chunks[0]['text'].split()
        words1 = chunks[1]['text'].split()
        overlap_ok = words0[-5:] == words1[:5]
        check(overlap_ok, "Overlap is correct", "Overlap words don't match between chunks")
else:
    print("  ❌ Function returned None — implement token_chunk()!")

## Exercise 4 ⭐⭐ — Manual TF-IDF Calculation

Calculate TF-IDF manually for a tiny corpus. This will solidify your understanding of the formula.

In [ ]:
# Given corpus
tiny_corpus = [
    "cat sat on mat",
    "cat sat on the hat",
    "the dog chased the cat",
]

# ─── Step 1: Compute TF for each document ────────────────────────────────────
def compute_tf(document: str) -> Dict[str, float]:
    """
    Compute Term Frequency for a single document.
    TF(t, d) = (count of t in d) / (total words in d)
    
    Returns dict: {word: tf_score}
    """
    # YOUR CODE HERE
    pass

# ─── Step 2: Compute IDF across corpus ───────────────────────────────────────
def compute_idf(corpus: List[str]) -> Dict[str, float]:
    """
    Compute Inverse Document Frequency for all terms in corpus.
    IDF(t) = log(N / df(t))   where df(t) = number of docs containing t
    Use math.log (natural log)
    
    Returns dict: {word: idf_score}
    """
    # YOUR CODE HERE
    pass

# ─── Step 3: Compute TF-IDF ───────────────────────────────────────────────────
def compute_tfidf(document: str, idf_scores: Dict[str, float]) -> Dict[str, float]:
    """
    Compute TF-IDF for each term in a document.
    TF-IDF(t, d) = TF(t, d) * IDF(t)
    """
    # YOUR CODE HERE
    pass


# ─── Run & Verify ─────────────────────────────────────────────────────────────
print("=" * 55)
print("Manual TF-IDF Calculation")
print("=" * 55)

idf = compute_idf(tiny_corpus) if compute_idf else {}

if idf:
    print("\nIDF Scores:")
    for word, score in sorted(idf.items(), key=lambda x: x[1], reverse=True):
        print(f"  {word:<10} {score:.4f}")
    
    print("\nTF-IDF for Document 1 ('cat sat on mat'):")
    tfidf_d1 = compute_tfidf(tiny_corpus[0], idf)
    if tfidf_d1:
        for word, score in sorted(tfidf_d1.items(), key=lambda x: x[1], reverse=True):
            print(f"  {word:<10} {score:.4f}")
    
    # Auto-check
    print()
    if idf:
        # 'the' appears in all 3 docs → IDF should be 0 (log(3/3)=0)
        the_idf = idf.get('the', None)
        check(the_idf is not None and abs(the_idf) < 0.01,
              f"IDF('the') ≈ 0 (got {the_idf:.4f})" if the_idf else "IDF('the') ≈ 0",
              f"IDF('the') should be ~0, got {the_idf}")
        
        # 'mat' appears in only 1 doc → higher IDF
        mat_idf = idf.get('mat', None)
        cat_idf = idf.get('cat', None)
        if mat_idf and cat_idf:
            check(mat_idf > cat_idf,
                  f"IDF('mat')={mat_idf:.3f} > IDF('cat')={cat_idf:.3f} (rarer=higher IDF)",
                  "'mat' is rarer than 'cat' so it should have higher IDF")
else:
    print("❌ Implement compute_tf, compute_idf, and compute_tfidf!")

## Exercise 5 ⭐⭐ — Cosine Similarity from Scratch

In [ ]:
def cosine_similarity_manual(vec_a: List[float], vec_b: List[float]) -> float:
    """
    Compute cosine similarity between two vectors.
    
    Formula: cos(θ) = (A · B) / (||A|| × ||B||)
    
    Steps:
    1. Compute dot product: sum(a_i * b_i)
    2. Compute magnitude of A: sqrt(sum(a_i^2))
    3. Compute magnitude of B: sqrt(sum(b_i^2))
    4. Return dot / (mag_a * mag_b), handle zero division
    """
    # YOUR CODE HERE
    pass


# ─── Test cases ───────────────────────────────────────────────────────────────
print("Testing cosine_similarity_manual():")

# Identical vectors → similarity = 1.0
a = [1, 2, 3]
b = [1, 2, 3]
r = cosine_similarity_manual(a, b)
check(r is not None and abs(r - 1.0) < 1e-6, 
      f"Identical vectors → 1.0 (got {r:.6f})",
      f"Identical vectors should give 1.0, got {r}")

# Orthogonal vectors → similarity = 0.0
c = [1, 0, 0]
d = [0, 1, 0]
r2 = cosine_similarity_manual(c, d)
check(r2 is not None and abs(r2) < 1e-6,
      f"Orthogonal vectors → 0.0 (got {r2})",
      f"Orthogonal vectors should give 0.0, got {r2}")

# Similar but not identical
e = [1, 1, 0]
f_ = [1, 0.9, 0.1]
r3 = cosine_similarity_manual(e, f_)
check(r3 is not None and 0.9 < r3 < 1.0,
      f"Similar vectors → high score (got {r3:.4f})",
      f"Should be between 0.9-1.0, got {r3}")

# Compare with sklearn
if r3:
    sklearn_score = cosine_similarity([e], [f_])[0][0]
    check(abs(r3 - sklearn_score) < 1e-6,
          f"Matches sklearn: {r3:.6f} ≈ {sklearn_score:.6f}",
          f"Doesn't match sklearn: yours={r3}, sklearn={sklearn_score}")

## Exercise 6 ⭐⭐ — Build a Keyword Search Engine

In [ ]:
class KeywordSearchEngine:
    """
    A simple keyword search engine over a document corpus.
    
    TODO: Implement the methods below.
    
    Required methods:
    - __init__()           : Initialize vectorizer and storage
    - add_documents()      : Fit TF-IDF on a list of (title, text) tuples
    - search()             : Return top-k results for a query
    - display_results()    : Pretty-print search results
    """
    
    def __init__(self):
        # Initialize TfidfVectorizer with ngram_range=(1,2) and stop_words='english'
        # Initialize storage for titles and texts
        # YOUR CODE HERE
        pass
    
    def add_documents(self, documents: List[Tuple[str, str]]):
        """
        Add documents to the search engine.
        
        Parameters:
        -----------
        documents : List of (title, text) tuples
        """
        # Store titles and texts separately
        # Fit TF-IDF vectorizer on texts
        # YOUR CODE HERE
        pass
    
    def search(self, query: str, top_k: int = 3) -> List[Dict]:
        """
        Search for relevant documents.
        
        Returns: List of dicts with 'rank', 'title', 'score', 'snippet'
        where 'snippet' is the first 150 characters of the text.
        """
        # Transform query using fitted vectorizer
        # Compute cosine similarity with all documents
        # Return top-k results (only those with score > 0)
        # YOUR CODE HERE
        pass
    
    def display_results(self, query: str, top_k: int = 3):
        """Pretty-print search results."""
        results = self.search(query, top_k)
        print(f"\n🔍 Search: '{query}'")
        print("─" * 60)
        if not results:
            print("  No results found.")
        for r in results:
            print(f"  [{r['rank']}] {r['title']}  (score: {r['score']:.4f})")
            print(f"      {r['snippet']}...")


# ─── Test corpus ──────────────────────────────────────────────────────────────
knowledge_base = [
    ("Python Basics", "Python is a high-level programming language known for its readability. It is widely used in data science and machine learning."),
    ("Machine Learning Intro", "Machine learning is a method of data analysis that automates analytical model building. It is based on the idea that systems can learn from data."),
    ("Neural Networks", "Neural networks are computing systems inspired by biological neural networks. Deep learning uses neural networks with many hidden layers."),
    ("RAG Systems", "Retrieval Augmented Generation combines a retrieval system with a language model to generate accurate and contextual responses."),
    ("Text Preprocessing", "Text preprocessing involves cleaning and transforming raw text data into a format suitable for NLP models including tokenization and stop word removal."),
    ("TF-IDF Explained", "TF-IDF stands for Term Frequency Inverse Document Frequency. It measures how important a word is to a document in a corpus."),
    ("Chunking Strategies", "Document chunking splits large texts into smaller pieces. Strategies include fixed-size, sentence-based, paragraph-based, and recursive chunking."),
    ("BM25 Algorithm", "BM25 is a ranking function used in search engines. It improves on TF-IDF by adding term saturation and document length normalization."),
]

engine = KeywordSearchEngine()
engine.add_documents(knowledge_base)

# Test searches
test_queries = [
    "how do neural networks work?",
    "what is BM25 and how does it differ from TF-IDF?",
    "python data science",
    "document splitting strategies"
]

for q in test_queries:
    engine.display_results(q, top_k=2)

## Exercise 7 ⭐⭐⭐ — Chunking Quality Evaluator

A good chunking strategy should produce chunks that:
1. Are not too short (< 30 words)
2. Are not too long (> 200 words for RAG)
3. Don't end mid-sentence
4. Have manageable size variance

In [ ]:
def evaluate_chunking(chunks: List[Dict], strategy_name: str = "Unknown") -> Dict:
    """
    Evaluate the quality of a chunking strategy.
    
    Compute these metrics:
    1. num_chunks     : Total number of chunks
    2. avg_words      : Average word count per chunk
    3. min_words      : Minimum word count
    4. max_words      : Maximum word count
    5. std_words      : Standard deviation of word counts (lower = more uniform)
    6. too_short      : % of chunks with < 30 words
    7. too_long       : % of chunks with > 200 words
    8. complete_sents : % of chunks ending with . ! or ? (proxy for sentence completeness)
    9. quality_score  : Composite score 0-100 (your formula, reward completeness + uniform size)
    
    Hints:
    - Access word count: len(chunk['text'].split())
    - Check ending: chunk['text'].strip()[-1] in '.!?'
    - For quality_score, penalize too_short, too_long, and high variance
      Example: quality_score = 100 - (too_short * 0.5) - (too_long * 0.5) - (std_words * 0.1)
    """
    # YOUR CODE HERE
    pass


# ─── Test on our document ─────────────────────────────────────────────────────
LONG_DOC = """
Artificial intelligence has transformed the modern world in remarkable ways. 
From voice assistants to medical diagnosis systems, AI is everywhere.

Machine learning, a core component of AI, enables computers to learn from experience.
Supervised learning uses labeled data to train models. Unsupervised learning finds 
hidden patterns in unlabeled data. Reinforcement learning trains agents through 
trial and error interaction with an environment.

Deep learning takes inspiration from the human brain's neural structure.
Convolutional neural networks excel at image recognition tasks.
Recurrent neural networks process sequential data like time series and text.
Transformer architectures have revolutionized natural language processing.

Large language models like GPT-4 and Claude are trained on trillions of tokens.
They can write code, answer questions, translate languages, and create content.
However, they sometimes produce incorrect information, a problem called hallucination.

Retrieval Augmented Generation addresses hallucination by grounding model outputs.
Instead of relying solely on memorized knowledge, RAG systems retrieve relevant 
documents at inference time and use them as context for generation.
This approach significantly improves factual accuracy and allows models to access 
up-to-date information beyond their training cutoff.
""".strip()

# Apply different strategies
def fixed_size_chunk_simple(text, size=200, overlap=30):
    chunks, start = [], 0
    while start < len(text):
        end = min(start+size, len(text))
        chunks.append({'id':len(chunks), 'text':text[start:end], 'size':end-start})
        start = end - overlap
    return chunks

def para_chunk_simple(text):
    paras = [p.strip() for p in re.split(r'\n\n+', text) if p.strip()]
    return [{'id':i,'text':p,'size':len(p)} for i,p in enumerate(paras)]

def sent_chunk_simple(text, n=3):
    sents = simple_sent_tokenize(text)
    chunks = []
    for i in range(0, len(sents), n):
        t = ' '.join(sents[i:i+n])
        chunks.append({'id':len(chunks),'text':t,'size':len(t)})
    return chunks

strategies = [
    ("Fixed-Size (200 chars)", fixed_size_chunk_simple(LONG_DOC, 200, 30)),
    ("Paragraph",             para_chunk_simple(LONG_DOC)),
    ("Sentence (3 sents)",    sent_chunk_simple(LONG_DOC, 3)),
]

print("Chunking Quality Evaluation")
print("=" * 70)

results = {}
for name, chunks in strategies:
    metrics = evaluate_chunking(chunks, name)
    results[name] = metrics
    if metrics:
        print(f"\n📊 {name}")
        print(f"   Chunks: {metrics.get('num_chunks')}  "
              f"Avg words: {metrics.get('avg_words', 0):.1f}  "
              f"Std: {metrics.get('std_words', 0):.1f}")
        print(f"   Too short: {metrics.get('too_short', 0):.1f}%  "
              f"Too long: {metrics.get('too_long', 0):.1f}%  "
              f"Complete sents: {metrics.get('complete_sents', 0):.1f}%")
        print(f"   Quality Score: {metrics.get('quality_score', 0):.1f}/100")
    else:
        print(f"\n❌ {name}: Implement evaluate_chunking()!")

---
# 🚀 Part 3 — Challenge Tasks

These are open-ended and harder. Try your best — there's no single right answer.

## Challenge 1 ⭐⭐⭐ — Multi-Document RAG Retriever

Build a retriever that can handle **multiple documents**, track **which document** each chunk came from, and support **hybrid scoring** (combine TF-IDF + BM25).

In [ ]:
# Multi-document knowledge base
MULTI_DOCS = {
    "ai_basics": """Artificial intelligence simulates human intelligence in machines. 
    AI systems can learn from experience, adjust to new inputs, and perform human-like tasks. 
    Key applications include speech recognition, image processing, and decision making.""",
    
    "ml_guide": """Machine learning is a type of AI that allows software to become more accurate 
    at predicting outcomes without being explicitly programmed to do so. 
    ML algorithms use historical data as input to predict new output values. 
    Common algorithms include linear regression, decision trees, and neural networks.""",
    
    "nlp_intro": """Natural language processing (NLP) combines computational linguistics with 
    statistical and machine learning models. NLP enables computers to process and understand 
    human language including speech and text. Applications include chatbots, translation, 
    sentiment analysis, and named entity recognition.""",
    
    "rag_paper": """Retrieval Augmented Generation (RAG) was introduced to address the limitations 
    of language models. RAG retrieves relevant documents before generating a response. 
    This grounds the generation in factual context, reducing hallucination significantly. 
    RAG systems typically use dense embeddings for retrieval but keyword methods also work well."""
}

class MultiDocRetriever:
    """
    A retriever that:
    1. Accepts multiple named documents
    2. Chunks each document using paragraph chunking
    3. Indexes all chunks with both TF-IDF and BM25
    4. Supports hybrid scoring: alpha * tfidf_score + (1-alpha) * bm25_score
    5. Returns results tagged with their source document name
    """
    
    def __init__(self, alpha: float = 0.5):
        """
        alpha : weight for TF-IDF (1-alpha goes to BM25). Default 0.5 = equal weight.
        """
        # YOUR CODE HERE
        pass
    
    def index(self, documents: Dict[str, str]):
        """Chunk and index all documents."""
        # YOUR CODE HERE
        pass
    
    def _bm25_scores(self, query: str) -> np.ndarray:
        """Compute BM25 scores (with simple implementation)."""
        # Use BM25FromScratch if you implemented it, or a simplified version
        # YOUR CODE HERE
        pass
    
    def retrieve(self, query: str, top_k: int = 3) -> List[Dict]:
        """
        Retrieve top-k chunks using hybrid scoring.
        Returns dicts with: rank, score, text, source_doc, chunk_id
        """
        # YOUR CODE HERE
        pass


# ─── Test ─────────────────────────────────────────────────────────────────────
retriever = MultiDocRetriever(alpha=0.6)  # 60% TF-IDF, 40% BM25
retriever.index(MULTI_DOCS)

queries = [
    "What is machine learning?",
    "How does RAG reduce hallucination?",
    "chatbots and NLP applications"
]

for q in queries:
    results = retriever.retrieve(q, top_k=2)
    print(f"\n🔍 '{q}'")
    if results:
        for r in results:
            print(f"   [{r['rank']}] [{r.get('source_doc','?')}] score={r['score']:.3f}: {r['text'][:80]}...")
    else:
        print("   ❌ Implement MultiDocRetriever methods!")

## Challenge 2 ⭐⭐⭐ — Preprocessing Impact Experiment

Run a **controlled experiment** to measure how each preprocessing step affects retrieval quality. Use the knowledge_base from Exercise 6.

In [ ]:
# Ground truth: which document SHOULD be the top result for each query
GROUND_TRUTH = {
    "how do neural networks learn": "Neural Networks",
    "what algorithm does elasticsearch use": "BM25 Algorithm",
    "splitting documents for language models": "Chunking Strategies",
    "term frequency in document corpus": "TF-IDF Explained",
    "python machine learning data": "Machine Learning Intro",
}

def evaluate_retrieval_accuracy(
    docs: List[Tuple[str, str]],
    ground_truth: Dict[str, str],
    preprocessor_fn = None,
    label: str = "No preprocessing"
) -> float:
    """
    Evaluate what % of queries return the correct top-1 result.
    
    Parameters:
    -----------
    docs          : List of (title, text) tuples
    ground_truth  : Dict mapping query → expected top-1 title
    preprocessor_fn : Optional function to preprocess texts before indexing
    label         : Name of this configuration
    
    Returns:
    --------
    Accuracy (0.0 to 1.0) = correct_top1 / total_queries
    """
    # YOUR CODE HERE
    # Steps:
    # 1. Apply preprocessor_fn to texts (if provided)
    # 2. Fit TF-IDF
    # 3. For each query in ground_truth:
    #    a. Retrieve top-1 document
    #    b. Check if it matches ground_truth[query]
    # 4. Return accuracy
    pass


# ─── Run experiments ─────────────────────────────────────────────────────────
def preprocess_v1(text): # Only lowercase
    return text.lower()

def preprocess_v2(text): # Lowercase + remove punctuation
    return re.sub(r'[^a-z0-9\s]', '', text.lower())

def preprocess_v3(text): # v2 + stop word removal
    tokens = re.findall(r'\b[a-z]+\b', text.lower())
    return ' '.join(t for t in tokens if t not in STOP_WORDS)

def preprocess_v4(text): # v3 + stemming
    tokens = re.findall(r'\b[a-z]+\b', text.lower())
    tokens = [t for t in tokens if t not in STOP_WORDS]
    return ' '.join(simple_stem(t) for t in tokens)

experiments = [
    ("No preprocessing",             None),
    ("Lowercase only",               preprocess_v1),
    ("Lowercase + no punctuation",   preprocess_v2),
    ("+ Stop word removal",          preprocess_v3),
    ("+ Stemming",                   preprocess_v4),
]

print("Preprocessing Impact on Retrieval Accuracy")
print("=" * 55)

accuracies = []
for label, fn in experiments:
    acc = evaluate_retrieval_accuracy(knowledge_base, GROUND_TRUTH, fn, label)
    if acc is not None:
        accuracies.append((label, acc))
        bar = '█' * int(acc * 20)
        print(f"  {label:<35} {acc*100:>5.1f}%  {bar}")

if accuracies:
    best = max(accuracies, key=lambda x: x[1])
    print(f"\n🏆 Best configuration: '{best[0]}' with {best[1]*100:.1f}% accuracy")
    print("\n💬 Discussion: What does this tell you about preprocessing for keyword retrieval?")
else:
    print("\n❌ Implement evaluate_retrieval_accuracy()!")

## Challenge 3 ⭐⭐⭐ — Chunk Size Sensitivity Analysis

How does chunk size affect retrieval quality? Find the **optimal chunk size** for our knowledge base.

In [ ]:
# Longer corpus for this experiment
LONG_CORPUS = """
Python is a high-level, interpreted programming language known for its clear syntax and readability.
It was created by Guido van Rossum and first released in 1991. Python supports multiple programming 
paradigms including procedural, object-oriented, and functional programming.

Machine learning is a branch of artificial intelligence focused on building systems that learn from 
data. Instead of following explicitly programmed instructions, these systems improve through experience.
Supervised learning, unsupervised learning, and reinforcement learning are the three main types.

Deep learning uses artificial neural networks with multiple layers to learn representations of data.
Convolutional neural networks excel at processing images. Recurrent neural networks handle sequential 
data. Transformer models, introduced in 2017, revolutionized natural language processing tasks.

Natural language processing enables computers to understand and generate human language.
Key NLP tasks include tokenization, part-of-speech tagging, named entity recognition, sentiment 
analysis, machine translation, and text summarization. BERT and GPT are prominent NLP models.

Text preprocessing is essential for NLP pipelines. It involves lowercasing, removing stop words,
stemming or lemmatization, and tokenization. The goal is to normalize text for consistent processing.
Different tasks require different preprocessing strategies.

Retrieval Augmented Generation combines document retrieval with text generation.
A retriever finds relevant documents from a knowledge base given a query.
The retrieved documents provide context to a language model for generating accurate answers.
This reduces hallucination and allows models to use up-to-date information.

TF-IDF stands for Term Frequency Inverse Document Frequency. It scores words by their importance 
in a document relative to the corpus. Words appearing in many documents get lower scores.
TF-IDF is widely used for information retrieval and text mining applications.

BM25 is an advanced ranking algorithm that improves upon TF-IDF. It adds term frequency 
saturation which limits the impact of very frequent terms. Document length normalization 
ensures fair scoring across documents of different lengths. BM25 is the default in Elasticsearch.
""".strip()

# YOUR TASK:
# 1. Implement chunk_and_retrieve() that:
#    - Chunks LONG_CORPUS with fixed_size_chunk at a given size
#    - Indexes with TF-IDF
#    - For each query in TEST_QUERIES, retrieves top-1
#    - Returns accuracy (% of queries where correct chunk was retrieved)
# 
# 2. Test chunk sizes: [100, 200, 300, 400, 500, 700]
# 3. Plot accuracy vs chunk_size

TEST_QUERIES_LONG = [
    ("Python programming language",     "Python"),
    ("supervised unsupervised learning", "Machine learning"),
    ("transformer neural networks NLP",  "Transformer"),
    ("TF-IDF term frequency scoring",    "TF-IDF"),
    ("BM25 elasticsearch ranking",       "BM25"),
]

def chunk_and_retrieve(text: str, chunk_size: int, queries: list) -> float:
    """
    Chunk text with fixed_size_chunk at chunk_size, index with TF-IDF,
    and check if each query retrieves a relevant chunk (one containing the keyword).
    
    Returns accuracy (0.0 - 1.0)
    """
    # YOUR CODE HERE
    pass


chunk_sizes = [50, 100, 150, 200, 300, 400, 500, 700]
accuracies  = []

for size in chunk_sizes:
    acc = chunk_and_retrieve(LONG_CORPUS, size, TEST_QUERIES_LONG)
    if acc is not None:
        accuracies.append(acc)
        print(f"Chunk size {size:>4}: {acc*100:.1f}% accuracy")

if accuracies:
    # Plot
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(chunk_sizes[:len(accuracies)], [a*100 for a in accuracies], 
            'o-', color='#2ecc71', linewidth=2, markersize=8)
    ax.fill_between(chunk_sizes[:len(accuracies)], [a*100 for a in accuracies], 
                    alpha=0.2, color='#2ecc71')
    ax.set_xlabel('Chunk Size (characters)')
    ax.set_ylabel('Retrieval Accuracy (%)')
    ax.set_title('Chunk Size vs Retrieval Accuracy')
    ax.grid(alpha=0.3)
    best_idx = np.argmax(accuracies)
    ax.axvline(chunk_sizes[best_idx], color='red', linestyle='--', label=f'Best: {chunk_sizes[best_idx]}')
    ax.legend()
    plt.tight_layout()
    plt.savefig('chunk_size_analysis.png', dpi=120)
    plt.show()
    print(f"\n🏆 Optimal chunk size: {chunk_sizes[best_idx]} characters")
else:
    print("\n❌ Implement chunk_and_retrieve()!")

## Challenge 4 ⭐⭐⭐ — Build Your Own Stopword List

Rather than using a pre-built stop word list, **derive** one automatically from a corpus using IDF scores.

In [ ]:
def auto_stopwords(
    corpus: List[str],
    max_df_ratio: float = 0.8,
    min_word_len: int = 2
) -> List[str]:
    """
    Automatically identify stop words from a corpus.
    
    Words that appear in more than max_df_ratio of documents are likely
    stop words (they appear everywhere and have no discriminative power).
    
    Parameters:
    -----------
    corpus        : List of text documents
    max_df_ratio  : Words appearing in > this fraction of docs = stop word
    min_word_len  : Ignore words shorter than this
    
    Returns:
    --------
    List of auto-detected stop words, sorted by document frequency (highest first)
    
    Steps:
    1. Tokenize each document (lowercase, alpha only)
    2. For each word, count how many documents it appears in
    3. Compute df_ratio = doc_count / len(corpus)
    4. Return words where df_ratio > max_df_ratio
    """
    # YOUR CODE HERE
    pass


# Test corpus
test_corpus = [
    "the machine learning model is very accurate and the results are good",
    "the neural network is trained on large datasets for better performance",
    "machine learning and deep learning are subfields of artificial intelligence",
    "the language model generates text and the output is very coherent",
    "retrieval systems find the most relevant documents for a given query",
    "the transformer architecture is the foundation of modern language models",
]

auto_stops = auto_stopwords(test_corpus, max_df_ratio=0.7)

if auto_stops is not None:
    print(f"Auto-detected stop words ({len(auto_stops)}):")
    print(auto_stops)
    
    # Compare with our manual list
    overlap = set(auto_stops) & STOP_WORDS
    new_words = set(auto_stops) - STOP_WORDS
    
    print(f"\nOverlap with manual stop words: {len(overlap)}")
    print(f"Overlap words: {sorted(overlap)}")
    print(f"\nNew words found automatically: {sorted(new_words)}")
    print("\n💬 Discussion: Are the automatically found stop words reasonable?")
    print("   Should 'learning' be a stop word if it appears in 4/6 docs?")
else:
    print("❌ Implement auto_stopwords()!")

---
# 📝 Part 4 — Reflection & Analysis

Answer these in your own words after completing the exercises.

### Reflection Q1
Which preprocessing step gave you the biggest surprise or most unexpected result? Why?

> YOUR ANSWER HERE

### Reflection Q2
After running Challenge 3 (chunk size analysis), what chunk size worked best? Why do you think smaller/larger chunks performed better or worse for your test queries?

> YOUR ANSWER HERE

### Reflection Q3
If TF-IDF and BM25 are both "keyword" methods, when would you prefer to use **semantic embeddings** instead? What are the fundamental limitations of keyword retrieval?

> YOUR ANSWER HERE

---
# ✅ Self-Grading Checklist

Run the cell below to see your completion status.

In [ ]:
print("Day 4 Workbook — Completion Checklist")
print("=" * 50)

tasks = [
    ("Q1-Q7: Concept questions answered",    True),   # Change to True when done
    ("Ex 1: clean_tweet() implemented",      False),
    ("Ex 2: top_n_words() implemented",      False),
    ("Ex 3: token_chunk() implemented",      False),
    ("Ex 4: TF-IDF from scratch",            False),
    ("Ex 5: cosine_similarity_manual()",     False),
    ("Ex 6: KeywordSearchEngine built",      False),
    ("Ex 7: evaluate_chunking() done",       False),
    ("Ch 1: MultiDocRetriever built",        False),
    ("Ch 2: Preprocessing impact experiment", False),
    ("Ch 3: Chunk size analysis completed",  False),
    ("Ch 4: auto_stopwords() implemented",   False),
    ("Reflections Q1-Q3 answered",           False),
]

done = sum(1 for _, v in tasks if v)
total = len(tasks)

for task, complete in tasks:
    icon = "✅" if complete else "⬜"
    print(f"  {icon}  {task}")

print(f"\n{'─'*50}")
print(f"  Completed: {done}/{total} ({done/total*100:.0f}%)")

if done == total:
    print("\n🎉 EXCELLENT! All tasks complete. Ready for Day 5!")
elif done >= total * 0.7:
    print(f"\n👍 Great progress! {total-done} more tasks to go.")
else:
    print(f"\n💪 Keep going! You've got this.")

print("\n📅 Next: Day 5 — Simulated Embeddings & Preparing Data for RAG Pipelines")